# Somo la 10 - Mawakala ya AI Katika Uzalishaji

Katika somo hili utajifunza **mifumo ya uzalishaji** kwa mawakala wa AI ukitumia Microsoft Agent Framework (Python). Tunashughulikia:

- **Ufuatiliaji** — kuongeza upimaji wa muda na uandishi wa kumbukumbu (logging) kwa mwingiliano wa mawakala
- **Tathmini** — kutumia wakala wa tathmini kupiga alama ubora wa majibu
- **Usimamizi wa gharama** — mikakati ya kuboresha tokeni na uchaguzi wa modeli

Muktadha ni **wakala wa usafiri** anayesaidia watumiaji kupanga safari, pamoja na ufuatiliaji na tathmini zilizowekwa juu yake.


## Usanidi


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Mambo ya Kuzingatia kwa Uzalishaji

Kuhamisha maajenti ya AI kutoka kwa mifano ya majaribio kwenda uzalishaji kunahitaji kutilia maanani kwa makini nguzo tatu:

1. **Uonekano** — Unahitaji ufahamu wa kile maajenti yanayofanya, muda yanachochukua, na ni zana gani yanazitumia. Bila ufuatiliaji na uandishi wa kumbukumbu, kutatua matatizo ya uzalishaji karibu haiwezekani.

2. **Tathmini** — Ukaguzi wa ubora wa kiotomatiki unahakikisha majibu ya maajenti yanabaki sahihi, kamili, na ya msaada kwa muda. Maajenti wa kutathmini yanaweza kutoa alama kwa majibu kulingana na vigezo vilivyowekwa.

3. **Usimamizi wa Gharama** — Matumizi ya tokeni yanaathiri moja kwa moja gharama. Mikakati kama kuboresha maagizo, uchaguzi wa modeli, na caching husaidia kudhibiti gharama bila kuathiri ubora.


## Kujenga Wakala Anayoweza Kufuatiliwa

Tunaelezea zana za usafiri na tunafunika wito la wakala kwa kipimo cha muda ili tuweze kufuatilia ucheleweshaji. Katika uzalishaji ungeunganisha na OpenTelemetry au backend ya ufuatiliaji inayofanana.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mifumo ya Tathmini

Mfumo wa uzalishaji unaotumika mara nyingi ni kutumia wakala wa pili kama **mtathmini**. Mtathmini huweka alama kwa majibu ya wakala mkuu kulingana na vigezo vilivyowekwa awali kama ukamilifu, usahihi, na jinsi yanavyosaidia.

Hii inaruhusu:
- Milango ya ubora ya kiotomatiki kabla majibu hayafikie watumiaji
- Ugundaji wa regression wakati maagizo au modeli zinapobadilika
- Ufuatiliaji endelevu wa utendaji wa wakala kwa muda


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mikakati ya Usimamizi wa Gharama

Kudhibiti gharama ni muhimu kwa mawakala wa AI wa uzalishaji. Hapa kuna mikakati muhimu:

| Strategy | Description |
|---|---|
| **Uboreshaji wa maagizo** | Weka maagizo ya mfumo mafupi. Ondoa muktadha unaorudiwa ili kupunguza tokeni za ingizo. |
| **Uchaguzi wa modeli** | Tumia modeli ndogo, nafuu (kwa mfano GPT-4o-mini) kwa kazi rahisi kama uainishaji au uchimbaji, na uhifadhi modeli kubwa kwa ajili ya hoja ngumu. |
| **Kuhifadhi (caching)** | Hifadhi matokeo ya zana na maswali yanayojitokeza mara kwa mara ili kuepuka kuita API zisizohitajika. |
| **Bajeti za tokeni** | Weka mipaka ya `max_tokens` ili kuzuia majibu marefu yasiyotegemewa. |
| **Kufanya kwa vikundi (batching)** | Unganisha maswali mengi ya mtumiaji katika wito mmoja wa API pale inavyowezekana. |

Kivitendo, mbinu ya ngazi hufanya kazi vyema: elekeza maombi rahisi kwa modeli ya haraka na nafuu, na peleka tu maswali changamano kwa modeli yenye uwezo zaidi.


## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Ongeza ufuatiliaji** kwa mwingiliano wa maajenti kwa kupima muda na kurekodi, na kuweka msingi wa kufuatilia na kusimamia.
2. **Pima majibu ya maajenti** kiotomatiki kwa kutumia wakala wa tathmini anayetoa alama kwa ukamilifu, usahihi, na msaada.
3. **Dhibiti gharama** kupitia uboreshaji wa maelekezo, uchaguzi wa modeli, uhifadhi wa cache, na bajeti za tokeni.

Mifano hii ya uzalishaji husaidia kuhakikisha maajenti yako wa AI ni wa kuaminika, wa kupimika, na gharama nafuu kwa kiwango kikubwa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tamko la kutokuwajibika**:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuhakikisha usahihi, tafadhali fahamu kuwa tafsiri za otomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Nyaraka ya awali katika lugha yake ya asili inapaswa kuzingatiwa kama chanzo chenye mamlaka. Kwa taarifa muhimu, tunapendekeza tafsiri ya kitaalamu inayofanywa na mtafsiri wa binadamu. Hatuwajibiki kwa maelewano potofu au tafsiri zisizo sahihi zitokanazo na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
